[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cuny-sps-datascience/citibike-data-pipeline/blob/main/teams/team-6/mvp/play_model1.ipynb)

# Play with Model 1 (calendar-only LightGBM)

Run cells top to bottom (Shift+Enter). Change the values in the **CHANGE ME** lines and re-run that cell.

- **1** load everything
- **2** pick a station + week → predicted vs actual
- **3** where does it miss? (error by hour of day)
- **4** worst days — the misses weather should fix
- **5** ask the model about any date/hour
- **6** experiment: turn features off, retrain, compare

**Run it:** click the Colab badge (nothing to install), or locally: `pip install -r requirements.txt` then `jupyter lab` from this folder.

*Before PR #72 is merged, use this Colab link instead:* [open from the PR branch](https://colab.research.google.com/github/jamilton08/citibike-data-pipeline/blob/jonathan/mvp-6-stations-model1/teams/team-6/mvp/play_model1.ipynb)

In [ ]:
# 0. Setup — only does something in Colab / when the data isn't next to the notebook
import os, shutil, subprocess, sys
if not os.path.exists('model1/predictions.parquet'):
    os.chdir('/content')
    target = '/content/cbp'
    sources = [('https://github.com/cuny-sps-datascience/citibike-data-pipeline.git', 'main'),
               ('https://github.com/jamilton08/citibike-data-pipeline.git', 'jonathan/mvp-6-stations-model1')]
    for repo, branch in sources:
        shutil.rmtree(target, ignore_errors=True)          # clear leftovers from earlier runs
        r = subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', branch, repo, target], capture_output=True, text=True)
        if r.returncode == 0 and os.path.exists(f'{target}/teams/team-6/mvp/model1/predictions.parquet'):
            print(f'using {branch} from {repo}')
            break
        print(f'MVP files not on {branch} yet, trying next source')
    else:
        raise RuntimeError('Could not find the MVP files on main or on the PR branch')
    os.chdir(f'{target}/teams/team-6/mvp')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm', 'pyarrow'], check=True)
print('working dir:', os.getcwd())

In [ ]:
# 1. Load data, model, predictions
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, lightgbm as lgb, matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
plt.rcParams['figure.figsize'] = (12, 4)
ZONE = 'America/New_York'

pred = pd.read_parquet('model1/predictions.parquet')
pred['target_local'] = pd.to_datetime(pred.target_hour_utc, utc=True).dt.tz_convert(ZONE)
booster = lgb.Booster(model_file='model1/lgbm_calendar.txt')
hourly = pd.read_parquet('data/trips_6stations_hourly.parquet')
hourly['hour_utc'] = pd.to_datetime(hourly.hour_utc, utc=True)

STATIONS = dict(pred.drop_duplicates('station_id')[['station_name','station_id']].astype(str).values)
print('stations:'); [print('  ', n) for n in STATIONS]
print('predictions:', len(pred), 'rows |', pred.split.value_counts().to_dict())

In [ ]:
# 2. One station, one week: actual vs LightGBM vs baseline
STATION = 'W 21 St & 6 Ave'      # CHANGE ME (any name printed above)
WEEK_START = '2026-06-08'        # CHANGE ME (a Monday between 2026-01-05 and 2026-08-24)

w = pred[(pred.station_name == STATION) & (pred.horizon == 1)]
w = w[(w.target_local >= pd.Timestamp(WEEK_START, tz=ZONE)) & (w.target_local < pd.Timestamp(WEEK_START, tz=ZONE) + pd.Timedelta(days=7))]
plt.bar(w.target_local, w.departures, width=1/24, color='lightgray', label='actual')
plt.plot(w.target_local, w.pred_lightgbm, lw=2, label='LightGBM')
plt.plot(w.target_local, w.pred_baseline, lw=1.5, ls='--', label='baseline (avg)')
plt.title(f'{STATION} — week of {WEEK_START}'); plt.ylabel('departures / hour'); plt.legend(); plt.show()
print(f'MAE this week — LightGBM {mean_absolute_error(w.departures, w.pred_lightgbm):.2f} | baseline {mean_absolute_error(w.departures, w.pred_baseline):.2f}')

In [ ]:
# 3. Where does it miss? Average error by hour of day (test period, all stations)
t = pred[(pred.split == 'test') & (pred.horizon == 1)].copy()
t['hour'] = t.target_local.dt.hour
t['err_lgbm'] = (t.pred_lightgbm - t.departures).abs()
t['err_base'] = (t.pred_baseline - t.departures).abs()
t.groupby('hour')[['err_lgbm','err_base']].mean().plot(kind='bar', width=0.8)
plt.title('Average miss by hour of day (test, Mar–Aug 2026)'); plt.ylabel('MAE (bikes)'); plt.xlabel('hour'); plt.show()
print('Rush hours = biggest misses. That is where weather and subway have the most room to help.')

In [ ]:
# 4. Worst days — biggest daily misses in the test period
d = t.assign(date=t.target_local.dt.date).groupby(['date','station_name']).agg(
        actual=('departures','sum'), predicted=('pred_lightgbm','sum')).reset_index()
d['miss'] = d.predicted - d.actual
d['miss_pct'] = (d.miss / d.predicted * 100).round(0)
print('Days the model OVER-predicted most (likely rain / cold / events):')
display(d.sort_values('miss', ascending=False).head(10).round(1))
print('Days it UNDER-predicted most (unexpectedly busy):')
display(d.sort_values('miss').head(10).round(1))
print('Look up a few of these dates — if the over-predicted ones were rainy, that is the case for model 2.')

In [ ]:
# 5. Ask the model: departures for any station / date / hour
STATION = 'Broadway & E 14 St'   # CHANGE ME
WHEN = '2026-07-15 17:00'        # CHANGE ME (local NYC time; any date works, even future)

HOLIDAYS = set(pd.to_datetime('''2024-01-01 2024-01-15 2024-02-19 2024-05-27 2024-06-19 2024-07-04 2024-09-02 2024-10-14 2024-11-11 2024-11-28 2024-12-25
2025-01-01 2025-01-20 2025-02-17 2025-05-26 2025-06-19 2025-07-04 2025-09-01 2025-10-13 2025-11-11 2025-11-27 2025-12-25
2026-01-01 2026-01-19 2026-02-16 2026-05-25 2026-06-19 2026-07-03 2026-09-07 2026-10-12 2026-11-11 2026-11-26 2026-12-25'''.split()).date)

def features_for(station_name, when_local, horizon=1):
    ts = pd.Timestamp(when_local, tz=ZONE) + pd.Timedelta(hours=horizon - 1)
    row = pd.DataFrame([{'station_id': STATIONS[station_name], 'horizon': horizon, 't_hour': ts.hour, 't_dow': ts.dayofweek,
                         't_month': ts.month, 't_doy': ts.dayofyear, 't_weekend': int(ts.dayofweek >= 5),
                         't_holiday': int(ts.date() in HOLIDAYS)}])
    row['station_id'] = pd.Categorical(row.station_id, categories=sorted(STATIONS.values()))
    return row, ts

for h in (1, 2, 3):
    X, ts = features_for(STATION, WHEN, h)
    print(f'{STATION} — {ts:%a %b %d %Y, %I %p}: ~{booster.predict(X)[0]:.1f} departures')
print('\nTry: a winter morning vs a summer evening, a weekend, a holiday like 2026-07-03.')

In [ ]:
# 6. Experiment: drop features, retrain, compare test MAE (~30-60 s per run)
data = pred.copy()  # rebuild train set from the hourly file
h = hourly.sort_values(['station_id','hour_utc'])
ALL = ['station_id','horizon','t_hour','t_dow','t_month','t_doy','t_weekend','t_holiday']

def build(hr):
    rows = []
    for hz in (1, 2, 3):
        f = hr[['station_id','hour_utc','departures']].copy()
        tgt = f.hour_utc + pd.Timedelta(hours=hz - 1)
        f = f.drop(columns='departures').merge(hr[['station_id','hour_utc','departures']].rename(columns={'hour_utc':'tgt'}),
                                              left_on=['station_id', tgt], right_on=['station_id','tgt'])
        f['horizon'] = hz; rows.append(f)
    f = pd.concat(rows); t = f.tgt.dt.tz_convert(ZONE)
    f['t_hour'], f['t_dow'], f['t_month'], f['t_doy'] = t.dt.hour, t.dt.dayofweek, t.dt.month, t.dt.dayofyear
    f['t_weekend'] = (f.t_dow >= 5).astype(int); f['t_holiday'] = t.dt.date.isin(HOLIDAYS).astype(int)
    f['date'] = t.dt.strftime('%Y-%m-%d'); f['station_id'] = f.station_id.astype(str).astype('category')
    return f
full = build(h)
tr = full[(full.date >= '2024-01-02') & (full.date <= '2025-12-31')]
va = full[(full.date >= '2026-01-01') & (full.date <= '2026-02-28')]
te = full[(full.date >= '2026-03-01') & (full.date <= '2026-08-31')]

def run(features):
    m = lgb.LGBMRegressor(objective='poisson', n_estimators=3000, learning_rate=0.03, num_leaves=63,
                          min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.9, verbose=-1)
    m.fit(tr[features], tr.departures, eval_set=[(va[features], va.departures)],
          callbacks=[lgb.early_stopping(150, verbose=False)])
    return mean_absolute_error(te.departures, m.predict(te[features]))

# CHANGE ME: add/remove experiments
experiments = {
    'all features':            ALL,
    'no day-of-year / month':  [f for f in ALL if f not in ('t_doy','t_month')],
    'no holiday':              [f for f in ALL if f != 't_holiday'],
    'hour + weekday only':     ['station_id','t_hour','t_dow'],
}
for name, feats in experiments.items():
    print(f'{name:28s} test MAE {run(feats):.3f}')
print('\nBaseline (station x hour x weekday avg) was 4.362.')